In [1]:
import numpy as np

In [2]:
class BoostingClassifier:

    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.models = []

    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y)

        y = np.where(y == 0, -1, 1)

        n_samples = X.shape[0]

        weights = np.ones(n_samples) / n_samples

        self.models = []

        for _ in range(self.n_estimators):

            best_error = float("inf")
            best_stump = None

            for feature in range(X.shape[1]):

                thresholds = np.unique(X[:, feature])

                for threshold in thresholds:

                    for polarity in [1, -1]:

                        predictions = np.ones(n_samples)

                        if polarity == 1:
                            predictions[X[:, feature] < threshold] = -1

                        else:
                            predictions[X[:, feature] >= threshold] = -1

                        error = np.sum(weights * (predictions != y))

                        if error < best_error:
                            best_error = error

                            best_stump = {
                                "feature": feature,
                                "threshold": threshold,
                                "polarity": polarity
                            }

            if best_error >= 0.5:
                break

            best_error = np.clip(best_error, 1e-10, 1 - 1e-10)

            alpha = 0.5 * np.log((1 - best_error) / best_error)

            self.models.append((best_stump, alpha))

            predictions = np.ones(n_samples)

            feature = best_stump["feature"]
            threshold = best_stump["threshold"]
            polarity = best_stump["polarity"]

            if polarity == 1:
                predictions[X[:, feature] < threshold] = -1

            else:
                predictions[X[:, feature] >= threshold] = -1

            weights *= np.exp(-alpha * y * predictions)

            weights /= np.sum(weights)

        return self

    def predict(self, X):

        X = np.array(X)

        final_predictions = np.zeros(X.shape[0])

        for stump, alpha in self.models:

            predictions = np.ones(X.shape[0])

            feature = stump["feature"]
            threshold = stump["threshold"]
            polarity = stump["polarity"]

            if polarity == 1:
                predictions[X[:, feature] < threshold] = -1

            else:
                predictions[X[:, feature] >= threshold] = -1

            final_predictions += alpha * predictions

        return np.where(final_predictions >= 0, 1, 0)

In [3]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

df["target"] = data.target

X = df.drop("target", axis=1)
y = df["target"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
boosting_clf = BoostingClassifier(n_estimators=50)

boosting_clf.fit(X_train, y_train)

y_pred = boosting_clf.predict(X_test)

In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))

print("Recall:", recall_score(y_test, y_pred))

print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.956140350877193
Precision: 0.958904109589041
Recall: 0.9722222222222222
F1 Score: 0.9655172413793104

Confusion Matrix:
[[39  3]
 [ 2 70]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

